In [ ]:
import re
from typing import TypedDict, cast

from datasets import DatasetDict, load_dataset
from vllm import LLM, SamplingParams

In [ ]:
class BaseDatasetEntry(TypedDict):
    question: str
    answer: str


class DatasetEntry(TypedDict):
    question: str
    answer: str
    gold: str

In [ ]:
def extract_gold(entry: BaseDatasetEntry) -> DatasetEntry:
    answer = entry["answer"]
    # Extract gold value from answer using regex
    match = re.search(r"#### (-?\d+)", answer)
    if match:
        gold = match.group(1)
        return DatasetEntry(question=entry["question"], answer=answer, gold=gold)
    else:
        raise ValueError("Could not extract gold value from answer: " + answer)


ds = cast(DatasetDict, load_dataset("gsm8k", "main"))

ds["train"] = ds["train"].map(extract_gold)
ds["test"] = ds["test"].map(extract_gold)

In [ ]:
engine = LLM(model="ibm-granite/granite-4.0-h-350m", gpu_memory_utilization=0.8)

In [ ]:
sampling_params = SamplingParams(temperature=1, max_tokens=2048)
tokenizer = engine.get_tokenizer()

In [ ]:
def chat_template(question: str):
    return [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer the question step by step and then give the final answer in the format '#### <answer>'.",
        },
        {"role": "user", "content": question},
    ]


outputs = engine.chat(
    messages=[chat_template(ds["test"][i]["question"]) for i in range(5)],
    sampling_params=sampling_params,
)

In [ ]:
# evaluate the model to see if it got the gold answer
correct = 0
total = 0

for i, output in enumerate(outputs):
    gold = ds["test"][i]["gold"]

    # print the generated text and header
    print(f"Q{i + 1:d}: {gold=} | prompt={ds['test'][i]['question']!r}")
    for o in output.outputs:
        print(f"Generated text: {o.text!r}")

    print()

In [ ]:
def reward_exact_answer(generated_text: str, gold: str) -> int:
    match = re.search(r"#### (-?\d+)", generated_text)
    if match:
        predicted = match.group(1)
        return 1 if predicted == gold else 0
    else:
        return 0


# print the generated text and reward the answer
for i, output in enumerate(outputs):
    gold = ds["test"][i]["gold"]

    # print the generated text and header
    print(f"Q{i + 1:d}: {gold=} | prompt={ds['test'][i]['question']!r}")
    for o in output.outputs:
        generated_text = o.text
        print(f"Generated text: {generated_text!r}")
        is_correct = reward_exact_answer(generated_text, gold)
        if is_correct:
            correct += 1
        else:
            print(f"Incorrect answer. Expected {gold}, but got {generated_text!r}")

    print()

In [ ]:
outputs = engine.chat(messages=[{"role": "user", "content": "hi"}])